# 1.4 · 子查询 & CTE / Subqueries & Common Table Expressions

> **课程定位 / Where this fits**
> **Part 1 第 4 课**。1.1-1.3 让你能在**一个查询里**搞定 SELECT/JOIN/GROUP BY。但真实业务的 SQL 经常需要**"查询的查询"**——比如"找出消费超过平均水平的客户"。这就是子查询和 CTE 的用武之地。
> **Part 1, lesson 4.** Real queries often need queries-of-queries — "customers whose spend is above average". Enter subqueries and CTEs.

> 📐 **符号约定 / Notation**
> 内层查询用 `q_inner`，外层查询用 `q_outer`。
> Inner query = nested; outer query = enclosing.

> 💡 **面试相关 / Interview-relevant**
> - "**Correlated** vs uncorrelated 子查询" ★★★★（必考）
> - "用 SQL 找每个组内的 top-3" ★★★★★（CTE + 窗口函数）
> - "递归 CTE 求阶乘 / 生成日期序列 / 员工层级" ★★★★
> - "子查询能否替换成 JOIN" ★★★
> - "CTE 和子查询性能差别" ★★★
>
> Top hits: correlated subquery, top-N per group, recursive CTE, subquery vs JOIN, CTE perf.

---

## 学习目标 / Learning Objectives

学完本节，你应该能：
After this notebook you'll be able to:

1. 区分 **scalar / row / table** 三种子查询的使用场景。
   Distinguish scalar / row / table subqueries.
2. 区分 **uncorrelated** vs **correlated** 子查询，并知道**性能差异的根源**。
   Tell apart uncorrelated and correlated subqueries; understand the perf gap.
3. 用 **CTE (`WITH`)** 把复杂 SQL 切成**可读的"局部变量"**，从上往下读。
   Use CTEs to make complex SQL top-down readable.
4. 写 **多 CTE 链**（`WITH a AS (...), b AS (...)`）。
   Write multi-CTE chains.
5. 用 **`WITH RECURSIVE`** 解决"生成序列"和"图遍历"问题。
   Use recursive CTEs for sequences and graph traversal.
6. 在子查询/CTE 之间做出正确选择，并写出**易调试、可复用**的 SQL。
   Pick the right tool between subquery / CTE / JOIN.

---

## 目录 / Table of Contents

1. [子查询的 3 种位置 / Three Subquery Positions](#1)
2. [Scalar Subquery（返回单值）](#2)
3. [`IN` / `EXISTS` 子查询（半连接）](#3)
4. [Subquery in `FROM`（派生表）](#4)
5. [Correlated Subquery ⭐ —— 每行重算](#5)
6. [CTE：`WITH` 语法 ⭐](#6)
7. [多 CTE 链 / Multiple CTEs](#7)
8. [`WITH RECURSIVE` ⭐ —— 递归 CTE](#8)
9. [CTE vs 子查询 vs JOIN —— 怎么选](#9)
10. [实战：复杂业务查询 / Hands-on](#10)
11. [小结 / Summary](#11)


<a id="1"></a>
## 1. 子查询的 3 种位置 / Three Subquery Positions

子查询 = **一个查询写在另一个查询的括号里**。按它**返回什么**和**写在哪**分三类：
Subquery = a query inside another query, in parens. Classified by what it returns and where it sits:

| 类型 / Type | 返回 / Returns | 典型位置 / Typical position |
|---|---|---|
| **Scalar** | 1 行 1 列 = 单值 | `SELECT`, `WHERE`, `HAVING`, `ON` 里的标量比较 |
| **Row**    | 1 行多列 | `WHERE (a, b) = (...)` 罕见 |
| **Table**  | 多行多列 | `FROM`（派生表）、`IN`、`EXISTS` |

### SQL 子查询的执行机制

```
外层 SQL: SELECT ... FROM ... WHERE col > (scalar子查询);

  ┌─ 执行内层一次 → 得到一个值 v
  └─ 用 v 替换 (子查询)，再执行外层
```

如果子查询用了**外层的列**（"correlated"），就**每行重算一遍内层**——慢得多但威力强。
If the subquery references an outer column ("correlated"), the inner runs once per outer row — slower but powerful.


In [ ]:
import duckdb
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

# 重建数据集 / Rebuild dataset
conn = duckdb.connect()
conn.sql("""
CREATE TABLE artist (artist_id INT PRIMARY KEY, name VARCHAR, country VARCHAR);
INSERT INTO artist VALUES
    (1,'The Beatles','UK'), (2,'Pink Floyd','UK'), (3,'Miles Davis','US'),
    (4,'Daft Punk','FR'), (5,'Radiohead','UK'), (6,'Anonymous Artist',NULL);

CREATE TABLE album (album_id INT PRIMARY KEY, title VARCHAR, artist_id INT, year INT);
INSERT INTO album VALUES
    (1,'Abbey Road',1,1969), (2,'The Dark Side of the Moon',2,1973),
    (3,'The Wall',2,1979), (4,'Kind of Blue',3,1959),
    (5,'Discovery',4,2001), (6,'Random Access Memories',4,2013),
    (7,'OK Computer',5,1997), (8,'Demos (unreleased)',6,2024);

CREATE TABLE track (
    track_id INT PRIMARY KEY, name VARCHAR, album_id INT,
    genre VARCHAR, seconds INT, price DECIMAL(4,2)
);
INSERT INTO track VALUES
    (1,'Come Together',1,'Rock',259,0.99), (2,'Something',1,'Rock',182,0.99),
    (3,'Here Comes the Sun',1,'Rock',185,0.99), (4,'Time',2,'Rock',413,1.29),
    (5,'Money',2,'Rock',382,1.29), (6,'Us and Them',2,'Rock',460,1.29),
    (7,'Another Brick in the Wall',3,'Rock',239,1.29),
    (8,'Comfortably Numb',3,'Rock',382,1.29),
    (9,'So What',4,'Jazz',545,1.49), (10,'Freddie Freeloader',4,'Jazz',586,1.49),
    (11,'Blue in Green',4,'Jazz',337,1.49),
    (12,'One More Time',5,'Electronic',320,1.29),
    (13,'Aerodynamic',5,'Electronic',213,1.29),
    (14,'Digital Love',5,'Electronic',301,1.29),
    (15,'Get Lucky',6,'Electronic',369,1.29),
    (16,'Instant Crush',6,'Electronic',337,1.29),
    (17,'Lose Yourself to Dance',6,'Electronic',353,1.29),
    (18,'Paranoid Android',7,'Rock',384,1.29),
    (19,'Karma Police',7,'Rock',261,1.29),
    (20,'No Surprises',7,'Rock',228,1.29),
    (21,'Untitled Demo 1',8,'Rock',180,0.50),
    (22,'Untitled Demo 2',8,'Rock',195,0.50);

CREATE TABLE customer (customer_id INT PRIMARY KEY, name VARCHAR, country VARCHAR, email VARCHAR);
INSERT INTO customer VALUES
    (1,'Alice Chen','US','alice@example.com'), (2,'Bob Smith','UK','BOB@example.com'),
    (3,'Charlie Davis','US','charlie@example.com'),
    (4,'Diana Park','DE','diana@example.com'), (5,'Ethan Miller','US','ethan@example.com'),
    (6,'Fiona Wong','JP',NULL);

CREATE TABLE invoice (
    invoice_id INT PRIMARY KEY, customer_id INT, track_id INT,
    invoice_date DATE, quantity INT
);
INSERT INTO invoice VALUES
    (1,1,1,DATE '2026-01-05',1), (2,1,9,DATE '2026-01-05',2),
    (3,2,4,DATE '2026-01-10',1), (4,2,18,DATE '2026-01-10',1),
    (5,3,5,DATE '2026-02-12',1), (6,3,6,DATE '2026-02-12',1),
    (7,3,7,DATE '2026-02-12',3), (8,4,15,DATE '2026-02-20',1),
    (9,4,12,DATE '2026-02-20',1), (10,4,13,DATE '2026-02-20',1),
    (11,5,9,DATE '2026-03-01',1), (12,5,10,DATE '2026-03-01',1),
    (13,5,11,DATE '2026-03-01',1), (14,5,4,DATE '2026-03-05',2),
    (15,1,18,DATE '2026-03-15',1), (16,1,19,DATE '2026-03-15',1),
    (17,2,15,DATE '2026-04-01',1), (18,3,14,DATE '2026-04-10',2),
    (19,4,8,DATE '2026-05-02',1), (20,5,1,DATE '2026-05-20',1);
""")

print(f"duckdb : {duckdb.__version__}")
print(f"tables : {conn.sql('SHOW TABLES').df()['name'].tolist()}")


<a id="2"></a>
## 2. Scalar Subquery / 标量子查询

返回 **1 个值**——可以当数字/字符串用在任何表达式里。
Returns a single value — usable anywhere a scalar can go.


In [ ]:
# 单值常量：歌曲的全店平均价 / "Average price across all tracks"
conn.sql("""
    SELECT AVG(price) AS overall_avg FROM track;
""").df()


In [ ]:
# 用作 WHERE 比较：高于平均价的歌 / Tracks priced above the global average
conn.sql("""
    SELECT name, price
    FROM track
    WHERE price > (SELECT AVG(price) FROM track)
    ORDER BY price DESC
    LIMIT 5;
""").df()


In [ ]:
# 用作 SELECT 里的"附加列"：每首歌 vs 全店平均的差额
# Use in SELECT as a constant column — "this track's price minus the global avg"
conn.sql("""
    SELECT
        name,
        price,
        (SELECT AVG(price) FROM track)                    AS overall_avg,
        ROUND(price - (SELECT AVG(price) FROM track), 3)  AS diff_from_avg
    FROM track
    ORDER BY diff_from_avg DESC
    LIMIT 5;
""").df()


**注意**：上面的标量子查询 `(SELECT AVG(price) FROM track)` 是 **uncorrelated**——它**只执行一次**（不依赖外层任何列），然后被当成常量。
The subquery here is **uncorrelated** — runs once, then used as a constant.

> ⚠ **scalar subquery 必须返回单值**：如果返回多行会报错。永远用 `LIMIT 1` 或聚合函数兜底。
> Scalar subqueries must return exactly one row — guard with `LIMIT 1` or an aggregate.


<a id="3"></a>
## 3. `IN` / `EXISTS` 子查询（半连接）/ Semi-join subqueries

### 3.1 `IN`：成员资格

```sql
SELECT ... WHERE col IN (SELECT ... FROM other);
```


In [ ]:
# 至少买过 1 首歌的客户 / Customers with ≥1 purchase
conn.sql("""
    SELECT name, country
    FROM customer
    WHERE customer_id IN (SELECT DISTINCT customer_id FROM invoice)
    ORDER BY name;
""").df()


### 3.2 `EXISTS`：存在性

```sql
SELECT ... WHERE EXISTS (SELECT 1 FROM ... WHERE outer_col = inner_col);
```

`EXISTS` 的内层子查询通常是 **correlated** 的（用了外层列）。它只关心"有没有匹配"，不取具体值，**一找到就返回 TRUE**。
`EXISTS` is usually correlated; it short-circuits to TRUE on first match.


In [ ]:
# 同样问题，用 EXISTS / Same question via EXISTS
conn.sql("""
    SELECT c.name, c.country
    FROM customer AS c
    WHERE EXISTS (
        SELECT 1 FROM invoice AS i
        WHERE i.customer_id = c.customer_id    -- ← correlated! 用了外层 c.customer_id
    )
    ORDER BY c.name;
""").df()


### 3.3 `IN` vs `EXISTS` 选哪个 / Pick which?

| 因素 / Factor | 倾向 / Prefer |
|---|---|
| 子查询有 **NULL** 风险 | **`EXISTS`** (`NOT IN + NULL` 会静默 0 行) |
| 子查询很大 + 外层很小 | **`EXISTS`**（短路）|
| 外层很大 + 子查询很小 | `IN`（可以转成 hash join）|
| **现代优化器** | 通常**等价处理**——选可读的 |

> 💡 一般工业建议：**默认用 `EXISTS`**，NULL-safe + 优化器友好。
> Industry default: prefer `EXISTS` — NULL-safe and optimizer-friendly.


<a id="4"></a>
## 4. Subquery in `FROM`（派生表 / Derived Table）

子查询写在 `FROM` 里就是"**派生表**"——临时给它一个别名，外层把它当成普通表用。
A subquery in `FROM` becomes a "derived table" — give it an alias, then treat it like any table.

```sql
SELECT outer.*
FROM (
    SELECT ... FROM ... WHERE ...
) AS inner_alias
JOIN ... ON ...;
```


In [ ]:
# 派生表：每个客户的"总消费"，然后筛 top-3
# Derived table: per-customer spend, then take top 3
conn.sql("""
    SELECT *
    FROM (
        SELECT
            c.name                              AS customer,
            ROUND(SUM(i.quantity * t.price), 2) AS spend
        FROM customer  AS c
        JOIN invoice   AS i USING (customer_id)
        JOIN track     AS t USING (track_id)
        GROUP BY c.customer_id, c.name
    ) AS per_customer
    ORDER BY spend DESC
    LIMIT 3;
""").df()


派生表能**多层嵌套**，但深度超过 2 层时**强烈建议用 CTE 替代**——读起来会清楚得多。
You can nest derived tables, but >2 levels = use CTEs for readability.


<a id="5"></a>
## 5. Correlated Subquery ⭐ —— 每行重算

**Correlated** = 子查询里用了外层的列。**每处理一行外层就执行一次内层**。
**Correlated** = inner uses outer column. **Inner runs once per outer row**.

这让它**威力巨大但很慢**。优化器会**尽力**把它改写成 JOIN，但不总成功。
Powerful but slow. Optimizers try to rewrite as JOINs but don't always succeed.

### 经典例题 / Classic problem

"对每个艺术家，列出他**最长**的那首歌"——这要"按艺术家分组的最大值"，但每行又要带回完整信息。
"For each artist, list their longest track" — needs per-artist max but also full row info.


In [ ]:
# Correlated subquery 写法 / Correlated subquery version
conn.sql("""
    SELECT
        ar.name           AS artist,
        t.name            AS longest_track,
        t.seconds
    FROM artist AS ar
    JOIN album  AS a  ON ar.artist_id = a.artist_id
    JOIN track  AS t  ON a.album_id = t.album_id
    WHERE t.seconds = (
        SELECT MAX(t2.seconds)
        FROM   track  AS t2
        JOIN   album  AS a2  ON t2.album_id = a2.album_id
        WHERE  a2.artist_id  = ar.artist_id   -- ← correlated! 引用外层 ar.artist_id
    )
    ORDER BY t.seconds DESC;
""").df()


**逐行执行**：对每个 artist，内层 `MAX(seconds)` 重新算一遍——`O(n × m)`。
For each outer artist row, the inner MAX runs again — `O(n × m)`.

**更现代的等价写法是窗口函数**（1.5 节会讲）：
The modern equivalent uses window functions (1.5):

```sql
SELECT artist, track, seconds
FROM (
    SELECT ar.name AS artist, t.name AS track, t.seconds,
           ROW_NUMBER() OVER (PARTITION BY ar.artist_id ORDER BY t.seconds DESC) AS rk
    FROM artist ar JOIN album a USING (artist_id) JOIN track t USING (album_id)
) sub WHERE rk = 1;
```

**性能**：窗口函数版本通常**显著更快**——只扫一次表，**避免了 O(n²)**。
The window-function version typically scans the table once → much faster.


<a id="6"></a>
## 6. CTE（Common Table Expression）⭐

CTE = `WITH alias AS (SELECT ...) SELECT ... FROM alias`。**让 SQL 从上往下读，像 Python 函数一样**。
CTE = `WITH alias AS (SELECT ...) SELECT ... FROM alias`. **Reads top-down like Python.**

### 6.1 基本语法 / Syntax

```sql
WITH per_customer AS (
    SELECT customer_id, SUM(quantity * price) AS spend
    FROM invoice i JOIN track t USING (track_id)
    GROUP BY customer_id
)
SELECT c.name, p.spend
FROM   customer AS c
JOIN   per_customer AS p USING (customer_id)
ORDER BY p.spend DESC;
```


In [ ]:
# Top-3 客户：CTE 版本（更易读）
# Top-3 customers with CTE — much more readable than nested derived
conn.sql("""
    WITH per_customer AS (
        SELECT
            customer_id,
            ROUND(SUM(i.quantity * t.price), 2) AS spend
        FROM invoice AS i
        JOIN track   AS t USING (track_id)
        GROUP BY customer_id
    )
    SELECT
        c.name      AS customer,
        p.spend
    FROM customer AS c
    JOIN per_customer AS p USING (customer_id)
    ORDER BY p.spend DESC
    LIMIT 3;
""").df()


### 6.2 CTE 不只是"派生表的好看版本" / CTEs aren't just pretty derived tables

CTE 的真正优势：
The real advantages:

1. **可读性 ⭐**——复杂查询从上往下读，每个 CTE 一句话表达"一个步骤"
2. **可复用**——同一个 CTE 在外层**多次引用**（派生表不行）
3. **可命名**——给业务步骤起名（`active_users` 比 `subquery_3` 易理解多了）
4. **WITH RECURSIVE** 是 SQL **唯一**的递归手段（下面讲）

> ⚠ **历史性能坑 / Historical perf trap**:
> PostgreSQL 11 以前**默认把 CTE 物化**（内存里存一份），有时比派生表慢。Postgres 12+ 改成 inline 优化。**MySQL 8 / DuckDB / Snowflake / BigQuery 都没这个问题**。
> Pre-PG-12 materialized CTEs by default; modern engines inline them.


<a id="7"></a>
## 7. 多 CTE 链 / Multiple CTEs

**`WITH` 后面可以跟多个 CTE，逗号分隔，后面的能引用前面的**。
Chain multiple CTEs with commas; later CTEs reference earlier ones.

```sql
WITH
    step1 AS (...),
    step2 AS (SELECT ... FROM step1 WHERE ...),
    step3 AS (SELECT ... FROM step2 JOIN ...)
SELECT * FROM step3;
```

**写复杂分析的标准结构** —— 像 Jupyter notebook 的 cells。
The standard structure for complex analytics — like notebook cells.


In [ ]:
# 复杂业务问题：每个 (genre, customer country) 组合的 top 客户 + 占比
# Complex Q: top customer per (genre × country) + their share of that segment's revenue
conn.sql("""
    WITH
    -- step 1: 每条 invoice 行的完整上下文
    enriched AS (
        SELECT
            c.customer_id, c.name AS cust_name, c.country AS cust_country,
            t.genre,
            i.quantity * t.price AS line_revenue
        FROM invoice  AS i
        JOIN customer AS c USING (customer_id)
        JOIN track    AS t USING (track_id)
    ),
    -- step 2: 每个 (genre, country, customer) 的消费
    by_segment_cust AS (
        SELECT
            genre, cust_country, customer_id, cust_name,
            SUM(line_revenue) AS spend
        FROM enriched
        GROUP BY genre, cust_country, customer_id, cust_name
    ),
    -- step 3: 每个 (genre, country) 的总消费
    by_segment AS (
        SELECT
            genre, cust_country,
            SUM(line_revenue) AS total_spend
        FROM enriched
        GROUP BY genre, cust_country
    )
    SELECT
        b.genre,
        b.cust_country                                  AS country,
        b.cust_name                                     AS top_customer,
        ROUND(b.spend, 2)                               AS customer_spend,
        ROUND(s.total_spend, 2)                         AS segment_total,
        ROUND(100.0 * b.spend / s.total_spend, 1)       AS pct_of_segment
    FROM by_segment_cust AS b
    JOIN by_segment      AS s USING (genre, cust_country)
    WHERE b.spend = (
        SELECT MAX(b2.spend)
        FROM by_segment_cust AS b2
        WHERE b2.genre = b.genre AND b2.cust_country = b.cust_country
    )
    ORDER BY pct_of_segment DESC;
""").df()


**结构清晰**：
- `enriched`：把 invoice 拍平成行级别（一行 = 一笔购买的全部信息）
- `by_segment_cust`：按 (genre × country × customer) 聚合
- `by_segment`：按 (genre × country) 算 segment 总额
- 主查询：取每个 segment 里 spend 最大的客户 + 算占比

Each CTE = one logical step, top to bottom — easier than 3-level nested derived tables.

**对比**：如果用嵌套子查询写同样的逻辑，最里层会 4 层，**几乎无法 debug**。
Equivalent nested-subquery version would be 4 levels deep — basically undebuggable.


<a id="8"></a>
## 8. `WITH RECURSIVE` ⭐ —— 递归 CTE

**SQL 唯一的递归工具**。语法：
SQL's only recursion mechanism:

```sql
WITH RECURSIVE rec_name AS (
    -- anchor: 初始结果集 / base case
    SELECT ...
    UNION ALL
    -- recursive: 引用 rec_name 自己 / step (refers to rec_name)
    SELECT ...
    FROM rec_name JOIN ... ON ...
    WHERE termination_condition       -- ⚠ 必须有终止条件，否则死循环
)
SELECT * FROM rec_name;
```

经典用例：
- 生成日期序列 / 数字序列
- 阶乘 / Fibonacci
- 员工层级（"找 Bob 下面所有人"）
- 类目树
- 图遍历

### 8.1 生成数字序列 / Number series


In [ ]:
# 生成 1..10 / Generate 1..10
conn.sql("""
    WITH RECURSIVE nums(n) AS (
        SELECT 1                       -- anchor
        UNION ALL
        SELECT n + 1 FROM nums         -- recursive
        WHERE n < 10                    -- termination ⭐
    )
    SELECT * FROM nums;
""").df()


**执行过程**：
1. anchor `SELECT 1` → 集合 `{1}`
2. 应用 recursive `{n+1 : n ∈ {1}, n < 10}` → `{2}`
3. 再应用 → `{3}` ... 直到 `n = 10` 后**条件失败**，递归结束
4. 输出 = anchor + 所有 recursive steps 的 UNION

### 8.2 生成日期序列 / Date series


In [ ]:
# 生成 2026-01-01 到 2026-01-10 每天 / All days from Jan 1 to Jan 10
conn.sql("""
    WITH RECURSIVE days(d) AS (
        SELECT DATE '2026-01-01'
        UNION ALL
        SELECT d + INTERVAL 1 DAY FROM days
        WHERE d < DATE '2026-01-10'
    )
    SELECT * FROM days;
""").df()


这是**填补"零销售日"的标准做法**：先生成全部日期 → LEFT JOIN 实际销售 → 缺日补 0。
The standard pattern for filling in zero-sale days.


In [ ]:
# 应用：每天销售额（含零销售日）
# Application: daily sales including zero-sale days
conn.sql("""
    WITH RECURSIVE days(d) AS (
        SELECT DATE '2026-01-01'
        UNION ALL
        SELECT d + INTERVAL 1 DAY FROM days WHERE d < DATE '2026-01-15'
    )
    SELECT
        days.d                                          AS day,
        COALESCE(ROUND(SUM(i.quantity * t.price), 2), 0) AS revenue
    FROM days
    LEFT JOIN invoice AS i ON i.invoice_date = days.d
    LEFT JOIN track   AS t USING (track_id)
    GROUP BY days.d
    ORDER BY days.d;
""").df()


**注意 2026-01-06 / 07 / 08 / 09 等都出现了**——即使没销售。
Note that even no-sale days appear with `revenue = 0`.

### 8.3 阶乘 / Factorial (just for fun)


In [ ]:
# 算 1..6 的阶乘 / Factorials of 1..6
conn.sql("""
    WITH RECURSIVE fact(n, val) AS (
        SELECT 1, CAST(1 AS BIGINT)
        UNION ALL
        SELECT n + 1, val * (n + 1) FROM fact
        WHERE n < 6
    )
    SELECT n, val AS factorial FROM fact;
""").df()


### 8.4 经典：员工层级 / Classic: employee hierarchy

```sql
WITH RECURSIVE org AS (
    -- anchor: 找 Bob
    SELECT id, name, manager_id, 0 AS depth
    FROM employee WHERE name = 'Bob'

    UNION ALL

    -- recursive: 找上一层的直接下属
    SELECT e.id, e.name, e.manager_id, org.depth + 1
    FROM employee e
    JOIN org ON e.manager_id = org.id
)
SELECT * FROM org;
```

这是**面试经典**——LinkedIn / Meta / Google 都问过。
A classic interview pattern.


<a id="9"></a>
## 9. 子查询 vs CTE vs JOIN —— 怎么选

| 场景 / Use case | 推荐 / Pick |
|---|---|
| **一个标量** 用在 SELECT/WHERE | **Scalar subquery** |
| 检查"存在性"（半连接）| `EXISTS` |
| 反连接（A 里 B 没有的）| `NOT EXISTS` 或 `LEFT JOIN + IS NULL` |
| **2 层以下**嵌套 / 临时一次性 | 派生表 / subquery in FROM |
| **3 层或以上**复杂逻辑 | **CTE** ⭐ |
| **同一个子结果多次用** | **CTE**（派生表必须重写）|
| 想取每组 top-N | **CTE + 窗口函数** ⭐ (1.5 学) |
| **递归 / 序列 / 层级** | **WITH RECURSIVE** |
| 把"先聚合再 JOIN" | CTE 或派生表 |
| 简单 N:1 关联取属性 | 普通 **JOIN** |

### 💡 一句话规则 / One-line rule

> 把"为啥这个查询要这么写"能用一句中文描述出来——**那一句就是 CTE 名字**。
> If you can describe "why this subquery" in one English sentence, that sentence is your CTE name.


<a id="10"></a>
## 10. 实战：复杂业务查询 / Hands-on

把本节学的工具综合应用——5 个真实业务问题。
Five realistic business questions exercising every construct.


In [ ]:
# Q1: 找出消费高于"全店客均"的客户 (scalar subquery)
# Customers spending above the overall per-customer average
conn.sql("""
    SELECT
        c.name,
        ROUND(SUM(i.quantity * t.price), 2) AS spend
    FROM customer AS c
    JOIN invoice  AS i USING (customer_id)
    JOIN track    AS t USING (track_id)
    GROUP BY c.customer_id, c.name
    HAVING SUM(i.quantity * t.price) > (
        SELECT AVG(per_cust.spend)
        FROM (
            SELECT SUM(i2.quantity * t2.price) AS spend
            FROM invoice AS i2
            JOIN track   AS t2 USING (track_id)
            GROUP BY i2.customer_id
        ) AS per_cust
    )
    ORDER BY spend DESC;
""").df()


In [ ]:
# Q2: 同样问题，CTE 版本 — 可读性更高 / Same Q with CTE
conn.sql("""
    WITH per_cust AS (
        SELECT
            customer_id,
            SUM(quantity * t.price) AS spend
        FROM invoice AS i JOIN track AS t USING (track_id)
        GROUP BY customer_id
    ),
    avg_spend AS (
        SELECT AVG(spend) AS avg_val FROM per_cust
    )
    SELECT
        c.name,
        ROUND(p.spend, 2) AS spend
    FROM per_cust AS p
    JOIN customer AS c USING (customer_id)
    CROSS JOIN avg_spend AS a
    WHERE p.spend > a.avg_val
    ORDER BY spend DESC;
""").df()


In [ ]:
# Q3: 每种 genre 销量最高的歌曲（correlated subquery）
# Best-selling track per genre — via correlated subquery
conn.sql("""
    SELECT
        t.genre,
        t.name                              AS top_selling_track,
        SUM(i.quantity)                     AS units_sold
    FROM track   AS t
    JOIN invoice AS i USING (track_id)
    GROUP BY t.genre, t.track_id, t.name
    HAVING SUM(i.quantity) = (
        SELECT MAX(unit_sum)
        FROM (
            SELECT t2.track_id, SUM(i2.quantity) AS unit_sum
            FROM track   AS t2
            JOIN invoice AS i2 USING (track_id)
            WHERE t2.genre = t.genre              -- ← correlated! reference outer t.genre
            GROUP BY t2.track_id
        )
    )
    ORDER BY units_sold DESC;
""").df()


In [ ]:
# Q4: 用 CTE 找出每月销售环比变化 (period-over-period)
# Month-over-month revenue change via CTE
conn.sql("""
    WITH monthly AS (
        SELECT
            DATE_TRUNC('month', invoice_date)   AS month,
            ROUND(SUM(i.quantity * t.price), 2) AS revenue
        FROM invoice AS i JOIN track AS t USING (track_id)
        GROUP BY 1
    )
    SELECT
        m1.month,
        m1.revenue                                    AS this_month,
        m2.revenue                                    AS prev_month,
        ROUND(m1.revenue - m2.revenue, 2)             AS delta,
        ROUND(100.0 * (m1.revenue - m2.revenue)
              / NULLIF(m2.revenue, 0), 1)             AS pct_change
    FROM monthly AS m1
    LEFT JOIN monthly AS m2
      ON m2.month = m1.month - INTERVAL 1 MONTH
    ORDER BY m1.month;
""").df()


In [ ]:
# Q5: 用 WITH RECURSIVE 生成"完整日期 × 客户"网格，找谁哪天买过 / 没买过
# Recursive: full (date × customer) grid → who shopped which day
conn.sql("""
    WITH RECURSIVE days(d) AS (
        SELECT DATE '2026-01-01'
        UNION ALL
        SELECT d + INTERVAL 1 DAY FROM days WHERE d < DATE '2026-05-31'
    ),
    -- 每个 (day × customer) 组合的实际购买
    daily_purchase AS (
        SELECT invoice_date, customer_id, COUNT(*) AS n
        FROM invoice GROUP BY 1, 2
    )
    SELECT
        days.d                          AS day,
        c.name                          AS customer,
        COALESCE(dp.n, 0)               AS purchases_that_day
    FROM days
    CROSS JOIN customer AS c
    LEFT JOIN daily_purchase AS dp
        ON dp.invoice_date = days.d AND dp.customer_id = c.customer_id
    WHERE c.customer_id = 1            -- 只看 Alice 演示 / show Alice only
    ORDER BY days.d
    LIMIT 15;
""").df()


<a id="11"></a>
## 11. 小结 / Summary

### 概念地图 / Concept map

```
子查询 (Subquery)
  │
  ├── 按返回行数
  │     ├── Scalar  (1 行 1 列)  → SELECT / WHERE / HAVING
  │     ├── Row     (1 行多列)
  │     └── Table   (多行多列)   → FROM / IN / EXISTS
  │
  ├── 按相关性
  │     ├── Uncorrelated  → 内层执行一次
  │     └── Correlated ⭐ → 每行外层执行一次内层
  │
  └── 何时改写成 CTE 或窗口函数

CTE (WITH)
  │
  ├── 单 CTE       WITH x AS (...) SELECT ... FROM x
  ├── 多 CTE 链    WITH a AS (...), b AS (... FROM a)
  └── WITH RECURSIVE ⭐
        ├── 必须 anchor + UNION ALL + recursive + 终止条件
        └── 用于：序列生成 / 阶乘 / 层级 / 图遍历
```

### 💡 必背 / Must-remember

| 规则 | 原因 |
|---|---|
| **scalar subquery 必须返回单值** | 否则报错或乱 |
| **`NOT IN` 怕 NULL** | 用 `NOT EXISTS` 兜底 |
| **3+ 层嵌套 → CTE** | 可读性 = 工业准则 |
| **recursive CTE 必须有终止条件** | 否则死循环 |
| **PG-11 之前 CTE 默认物化** | 现在不用担心了 |
| **`EXISTS` 短路 → 通常比 `IN` 快** | 优化器友好 |

### 💡 工业速查 / Industry cheat sheet

```sql
-- 标量子查询：和全店均值比 / Compare to global avg
SELECT name, price FROM track WHERE price > (SELECT AVG(price) FROM track);

-- 存在性：买过任何东西的客户 / Has any purchase
SELECT * FROM customer c
WHERE EXISTS (SELECT 1 FROM invoice WHERE customer_id = c.customer_id);

-- 派生表 → CTE 改写 / Derived → CTE
WITH per_user AS (SELECT user_id, COUNT(*) c FROM events GROUP BY user_id)
SELECT * FROM per_user WHERE c > 100;

-- 递归：生成日期序列 / Generate date series
WITH RECURSIVE d(x) AS (
    SELECT DATE '2026-01-01'
    UNION ALL
    SELECT x + INTERVAL 1 DAY FROM d WHERE x < DATE '2026-12-31'
) SELECT * FROM d;

-- Top-N per group 模板（CTE + correlated）/ Top-N per group (CTE)
WITH ranked AS (
    SELECT t.*, ROW_NUMBER() OVER (PARTITION BY g ORDER BY metric DESC) AS r
    FROM t
)
SELECT * FROM ranked WHERE r <= 3;     -- 真正惯用是窗口函数！1.5 节
```

### 💡 面试速查 / Interview must-knows

1. **Correlated vs Uncorrelated**：内层是否用了外层列
2. **Correlated 每行重算** → 慢 → 优先用窗口函数 / 改写成 JOIN
3. **`EXISTS` 比 `IN` 通常更安全** —— NULL 不会坑你
4. **CTE = 可读性 + 复用 + 递归**
5. **WITH RECURSIVE 三部曲**：anchor + UNION ALL + recursive + termination
6. **生成日期序列** 是面试经典 → recursive CTE 一行搞定

### 下一节预告 / Next up

**Part 1.5 · 窗口函数** —— `ROW_NUMBER` / `RANK` / `DENSE_RANK` / `LAG` / `LEAD` / 累计 / 滑动窗口。一旦学会，**你会发现这一节的 correlated 子查询绝大多数能被一行窗口函数替代**。
**Part 1.5 · Window Functions** — `ROW_NUMBER / RANK / LAG / LEAD` / running aggregates. Once you know these, most correlated subqueries can be replaced by a one-liner.
